 # **Lab 04 – Spark Streaming**

## **Group: CDHD**

### **Team Members**

| STT | Full Name           | Student ID |
| :-: | ------------------- | :--------: |
|  1  | Phan Thị Phương Chi |  23120025  |
|  2  | Trần Thanh Đạt      |  23120030  |
|  3  | Phạm Ngọc Duy       |  23120035  |
|  4  | Lê Hoàng Mỹ Hạ      |  23120038  |

## **Task 1: Repository Cloning and File Discovery**

### Mục tiêu
- Khám phá toàn bộ kho mã nguồn Python được chọn làm **corpus** cho CPG Parser Service (Task 2).
- Kết quả của task này là danh sách file `.py` đã phân loại, lưu vào `output/discovered_files.csv`. 
  
### Thông tin repo

| Trường        | Giá trị |
|:--------------|:--------|
| **Tên repo**  | `peft` |
| **Tổ chức**   | `huggingface` |
| **URL clone** | `https://github.com/huggingface/peft.git` |

### 0. Imports & cấu hình đường dẫn

In [1]:
import os
import sys
import subprocess
import platform
from pathlib import Path

import pandas as pd

# Thư mục gốc của project (thư mục chứa notebook)
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "peft").exists() and (PROJECT_ROOT.parent / "peft").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Đường dẫn repository và thư mục output trên Windows
WIN_REPO_PATH = str(PROJECT_ROOT / "peft")
WIN_OUTPUT_DIR = str(PROJECT_ROOT / "output")

# Đường dẫn tương ứng khi chạy trong WSL
WSL_REPO_PATH = "/mnt/d/Old/spark-streaming-lab/peft"
WSL_OUTPUT_DIR = "/mnt/d/Old/spark-streaming-lab/output"

REPO_CLONE_URL = "https://github.com/huggingface/peft.git"

# Chọn đường dẫn phù hợp với hệ điều hành hiện tại
if platform.system() == "Windows":
    REPO_ROOT = Path(WIN_REPO_PATH)
    OUTPUT_DIR = Path(WIN_OUTPUT_DIR)
else:
    REPO_ROOT = Path(WSL_REPO_PATH)
    OUTPUT_DIR = Path(WSL_OUTPUT_DIR)

# Tạo thư mục output nếu chưa tồn tại
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python     : {sys.version}")
print(f"Hệ điều hành: {platform.system()} - {platform.version()}")
print(f"PROJECT    : {PROJECT_ROOT}")
print(f"REPO_ROOT  : {REPO_ROOT}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")

# Thêm thư mục project vào PYTHONPATH để có thể import các module trong src
sys.path.insert(
    0,
    str(Path(os.path.abspath(".")).parent.parent)
    if Path(os.path.abspath(".")).name == "task1" and Path(os.path.abspath(".")).parent.name == "src"
    else str(Path(os.path.abspath(".")))
)

from src.task1.clone import clone_repo_if_needed
from src.task1.discover import (
    is_auto_generated,
    classify_file,
    count_lines,
    scan_repo,
    build_dataframe,
    smoke_test,
)

Python     : 3.11.5 | packaged by conda-forge | (main, Aug 27 2023, 03:23:48) [MSC v.1936 64 bit (AMD64)]
Hệ điều hành: Windows - 10.0.26200
PROJECT    : D:\_STUDY\BigData\Lab\Lab4\code\spark-streaming-lab
REPO_ROOT  : D:\_STUDY\BigData\Lab\Lab4\code\spark-streaming-lab\peft
OUTPUT_DIR : D:\_STUDY\BigData\Lab\Lab4\code\spark-streaming-lab\output


### 1. Kiểm tra & Clone Repository

In [2]:
clone_repo_if_needed(REPO_ROOT, REPO_CLONE_URL)

assert REPO_ROOT.exists(), f"[LỖI] Không tìm thấy repository tại {REPO_ROOT}"
print(f"[OK] Repository hợp lệ: {REPO_ROOT}")

[INFO] Repository đã tồn tại tại: D:\_STUDY\BigData\Lab\Lab4\code\spark-streaming-lab\peft
[INFO] Bỏ qua bước clone.
[OK] Repository hợp lệ: D:\_STUDY\BigData\Lab\Lab4\code\spark-streaming-lab\peft


### 2. Duyệt đệ quy & thu thập tất cả file `.py`

In [3]:
all_py_files_relative, categories = scan_repo(REPO_ROOT)

print("\nVí dụ 10 tệp đầu tiên:")
for p in all_py_files_relative[:10]:
    print(f"  {p}")

[OK] Tìm thấy 431 tệp .py

Ví dụ 10 tệp đầu tiên:
  docs\source\_config.py
  examples\adamss_finetuning\glue_adamss_asa_example.py
  examples\adamss_finetuning\glue_adamss_asa_manual_example.py
  examples\adamss_finetuning\image_classification_adamss_asa.py
  examples\adamss_finetuning\test_adamss_quick.py
  examples\alora_finetuning\alora_finetuning.py
  examples\arrow_multitask\arrow_phi3_mini.py
  examples\bdlora_finetuning\chat.py
  examples\beft_finetuning\beft_finetuning.py
  examples\boft_controlnet\__init__.py


### 3. Phân loại file `.py`

**Mục tiêu:** Loại bỏ các file không liên quan trước khi đưa vào CPG Parser, giúp giảm nhiễu
và tăng chất lượng đồ thị phụ thuộc.

**Bốn nhóm phân loại và lý do loại trừ:**

| Nhóm | Tiêu chí | Lý do loại trừ |
|:-----|:---------|:---------------|
| **test** | Thư mục chứa `test`/`tests`; tên file bắt đầu bằng `test_` | Test code không đại diện cho logic nghiệp vụ, chứa mock và fixture gây nhiễu CPG |
| **setup** | Tên file chính xác là `setup.py`, `__init__.py`, `conftest.py` (ngoài thư mục test) | File scaffolding/packaging, không chứa logic domain |
| **auto-generated** | Nằm trong `build/`, `dist/`, `*.egg-info/`; hoặc header chứa `DO NOT EDIT` / `auto-generated` | Code sinh tự động thường bị flatten, không phản ánh cấu trúc thiết kế thực |
| **source** | Tất cả file còn lại | Corpus chính cho CPG Parser Service ở Task 2 |

**Thứ tự ưu tiên phân loại:** `auto-generated` -> `setup` (exact filename) -> `test` (path/prefix) -> `source`

In [4]:
from collections import Counter

category_counts = Counter(categories)

print("=" * 50)
print("KẾT QUẢ PHÂN LOẠI TỆP .py")
print("=" * 50)

for cat in ["source", "test", "setup", "auto-generated"]:
    print(f"  {cat:<20}: {category_counts.get(cat, 0):>5} tệp")

print("-" * 50)
print(f'  {"TỔNG":<20}: {sum(category_counts.values()):>5} tệp')

# Kiểm tra conftest.py phải được phân loại là setup
conftest = [
    (p, c)
    for p, c in zip(all_py_files_relative, categories)
    if p.name == "conftest.py"
]

print("\nKiểm tra conftest.py (phải là setup):")

for path, cat in conftest[:5]:
    status = "OK" if cat == "setup" else "LỖI"
    print(f"  [{status}] {path} -> {cat}")

KẾT QUẢ PHÂN LOẠI TỆP .py
  source              :   309 tệp
  test                :    63 tệp
  setup               :    59 tệp
  auto-generated      :     0 tệp
--------------------------------------------------
  TỔNG                :   431 tệp

Kiểm tra conftest.py (phải là setup):
  [OK] tests\conftest.py -> setup


**Kiểm chứng heuristic is_auto_generated() (Smoke Test)**  
Do repository ban đầu không chứa các file sinh tự động nên kết quả phát hiện là 0 file. Để kiểm tra hàm `is_auto_generated()`, nhóm tạo một số file và thư mục giả lập (ví dụ build/, dist/,...) rồi chạy lại chương trình. Nếu các file này được phát hiện đúng là auto-generated thì có thể kết luận detector hoạt động như mong đợi. Sau khi kiểm tra, nhóm xóa các file giả lập để giữ nguyên trạng thái của repository.

In [5]:
smoke_test()

  [OK] Tệp có 'DO NOT EDIT'
  [OK] Tệp Python thông thường
  [OK] Đường dẫn chứa 'build/'
[OK] Smoke test thành công (3/3)


### 4. Tạo DataFrame & Lưu ra CSV

In [6]:
df = build_dataframe(REPO_ROOT, all_py_files_relative, categories, OUTPUT_DIR)

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 120)

print("\n10 dòng đầu:")
df.head(10)

[OK] Đã lưu tệp CSV: D:\_STUDY\BigData\Lab\Lab4\code\spark-streaming-lab\output\discovered_files.csv (431 dòng)

10 dòng đầu:


,relative_path,size_bytes,category,num_lines
0,docs/source/_config.py,287,source,7
1,examples/adamss_finetuning/glue_adamss_asa_example.py,15224,source,382
2,examples/adamss_finetuning/glue_adamss_asa_manual_example.py,16825,source,411
3,examples/adamss_finetuning/image_classification_adamss_asa.py,17368,source,452
4,examples/adamss_finetuning/test_adamss_quick.py,4587,test,176
5,examples/alora_finetuning/alora_finetuning.py,9839,source,259
6,examples/arrow_multitask/arrow_phi3_mini.py,14755,source,383
7,examples/bdlora_finetuning/chat.py,1972,source,64
8,examples/beft_finetuning/beft_finetuning.py,4083,source,122
9,examples/boft_controlnet/__init__.py,0,setup,0


In [7]:
summary_df = (
    df.groupby("category")
      .agg(
          num_files=("relative_path", "count"),
          total_bytes=("size_bytes", "sum"),
          total_lines=("num_lines", "sum"),
          avg_lines=("num_lines", "mean"),
      )
      .reset_index()
      .sort_values("num_files", ascending=False)
)

summary_df["avg_lines"] = summary_df["avg_lines"].round(1)

print("Thống kê theo từng loại tệp:")
summary_df

Thống kê theo từng loại tệp:


,category,num_files,total_bytes,total_lines,avg_lines
1,source,309,4055581,93559,302.8
2,test,63,2085829,48547,770.6
0,setup,59,75046,2150,36.4


### 5. Bảng tổng kết cuối cùng (Lab Report)

In [8]:
total_py_files = len(df)
n_test         = category_counts.get('test', 0)
n_setup        = category_counts.get('setup', 0)
n_autogen      = category_counts.get('auto-generated', 0)
n_excluded     = n_test + n_setup + n_autogen
n_source       = category_counts.get('source', 0)

print('╔══════════════════════════════════════════════════════╗')
print('║           TASK 1 – FINAL SUMMARY (Lab Report)        ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  Repository : huggingface/peft                       ║')
print(f'║  URL        : https://github.com/huggingface/peft    ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  Total .py files discovered        : {total_py_files:>5}           ║')
print('╠──────────────────────────────────────────────────────╣')
print(f'║  Excluded – test files             : {n_test:>5}           ║')
print(f'║  Excluded – setup/init files       : {n_setup:>5}           ║')
print(f'║  Excluded – auto-generated files   : {n_autogen:>5}           ║')
print(f'║  Total excluded                    : {n_excluded:>5}           ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  SOURCE files -> CPG Parser Service : {n_source:>5}          ║')
print('╚══════════════════════════════════════════════════════╝')

╔══════════════════════════════════════════════════════╗
║           TASK 1 – FINAL SUMMARY (Lab Report)        ║
╠══════════════════════════════════════════════════════╣
║  Repository : huggingface/peft                       ║
║  URL        : https://github.com/huggingface/peft    ║
╠══════════════════════════════════════════════════════╣
║  Total .py files discovered        :   431           ║
╠──────────────────────────────────────────────────────╣
║  Excluded – test files             :    63           ║
║  Excluded – setup/init files       :    59           ║
║  Excluded – auto-generated files   :     0           ║
║  Total excluded                    :   122           ║
╠══════════════════════════════════════════════════════╣
║  SOURCE files -> CPG Parser Service :   309          ║
╚══════════════════════════════════════════════════════╝


### 6. Reflection

**What worked**
- `git clone --depth 1` giúp lấy repo `peft` nhanh và nhẹ, đủ dùng cho việc phân tích tĩnh mã nguồn.
- `pathlib.Path.rglob("*.py")` duyệt đệ quy toàn bộ 431 file `.py` mà không cần xử lý riêng cho từng hệ điều hành.
- Heuristic phân loại 4 nhóm (`auto-generated` -> `setup` -> `test` -> `source`) cho kết quả nhất quán, đã kiểm chứng bằng smoke test riêng cho `is_auto_generated()` (3/3 test case PASS).

**What failed**
- Bản đầu tiên đặt thứ tự ưu tiên là `test` trước `setup`, khiến `conftest.py` bị nhận nhầm thành `test` (do `"test"` là substring của `"conftest"`). Notebook đã bổ sung một bước kiểm tra riêng (in ra danh sách `conftest.py` và category tương ứng) để phát hiện lỗi này.
- Repo `peft` không có sẵn file/thư mục auto-generated (`build/`, `dist/`, `.egg-info/`) nên nhóm không thể xác nhận `is_auto_generated()` hoạt động đúng chỉ bằng cách chạy trên repo thật.

**How resolved**
- Đổi thứ tự ưu tiên phân loại: kiểm tra khớp tên file chính xác (`setup`) trước khi kiểm tra từ khoá `test`, giải quyết dứt điểm lỗi `conftest.py`.
- Viết smoke test tạo file/thư mục giả lập (có marker `DO NOT EDIT`, path chứa `build/`) để kiểm chứng `is_auto_generated()` hoạt động đúng, bù lại việc repo thật không có ca auto-generated nào.

## **Task 2: Incremental CPG Parser Service**

### Mục tiêu
- Xây dựng **Parser Service** (`cpg_parser.py`) xử lý file Python **từng file một** (không parse cả repo một lần).
- Dùng thư viện **`ast`** (Python standard library) để trích xuất AST nodes, CFG edges, DFG edges, và CALL edges.
- Mỗi phần tử có **ID ổn định** (UUIDv5) để reprocess không tạo trùng downstream.
- Emit 3 luồng sự kiện chính: **nodes**, **edges**, **metadata** (+ `errors` khi parse lỗi).
- Service hoạt động với **bounded memory**: parse 1 file → emit → flush → sang file tiếp theo.

### Thiết kế nhanh

| Thành phần | Cách làm |
|:---|:---|
| Parser | `ast.parse` + `ast.NodeVisitor` trong `src/task2/cpg_parser.py` |
| Stable node ID | `uuid.uuid5(NAMESPACE_DNS, f"{file_path}:{ast_path}")` |
| Stable edge ID | `uuid.uuid5(NAMESPACE_DNS, f"{src}->{tgt}:{edge_type}")` |
| Metadata ID | `uuid.uuid5(NAMESPACE_DNS, file_path)` |
| Input | `output/discovered_files.csv` (chỉ `category == source`) |
| Chế độ chạy | `--dry-run` (không Kafka) / emit Kafka (`--bootstrap-servers`) |


### 0. Cấu hình

Import các module trong `src/task2/` — logic parser nằm trong `src/task2/cpg_parser.py`.

In [9]:
from src.task2.dry_run import dry_run
from src.task2.verify_stable_id import test_stable_id
from src.task2.config import PARSER_SCRIPT, DISCOVERED_CSV, REPO_ROOT, DEMO_LIMIT

print(f"Parser script : {PARSER_SCRIPT}")
print(f"Discovered CSV: {DISCOVERED_CSV}")
print(f"Repo root     : {REPO_ROOT}")
print(f"Demo limit    : {DEMO_LIMIT} files")

Parser script : D:\_STUDY\BigData\Lab\Lab4\code\spark-streaming-lab\src\task2\cpg_parser.py
Discovered CSV: D:\_STUDY\BigData\Lab\Lab4\code\spark-streaming-lab\output\discovered_files.csv
Repo root     : D:\_STUDY\BigData\Lab\Lab4\code\spark-streaming-lab\peft
Demo limit    : 5 files


### 1. Dry-run Parser Service (không gửi Kafka)

Chạy `src/task2/dry_run.py` ở chế độ `--dry-run` trên 5 file source đầu tiên.

In [10]:
def run_task2_dry_run():
    ok = dry_run()
    if not ok:
        print('[CANH BAO] Dry-run khong thanh cong — kiem tra discovered_files.csv va thu muc peft/')

run_task2_dry_run()

[INFO] Bat dau dry-run Parser Service tren 5 files...
=== Starting CPG Parser Service ===
Discovered CSV : D:\_STUDY\BigData\Lab\Lab4\code\spark-streaming-lab\output\discovered_files.csv
Repo Root      : D:\_STUDY\BigData\Lab\Lab4\code\spark-streaming-lab\peft
Kafka Brokers  : localhost:9092
Schema Version : 1.0.0
Dry Run Mode   : True

Found 5 source files to parse.
[1/5] Processing: docs\source\_config.py ... Parsed successfully (Nodes: 5, Edges: 6) [DRY RUN]

--- SAMPLE NODE EVENT ---
{
  "id": "b5453b47-07b0-5ad7-beb3-42b79396bb76",
  "type": "Module",
  "label": "Module",
  "file_path": "docs/source/_config.py",
  "ast_path": "Module@0:0-0:0",
  "start_line": null,
  "start_column": null,
  "end_line": null,
  "end_column": null,
  "code": "",
  "properties": {},
  "schema_version": "1.0.0",
  "event_time": "2026-07-24T13:00:05.907274Z"
}

--- SAMPLE EDGE EVENT ---
{
  "id": "5a01cb7a-4332-5df7-8964-570b8f7935a6",
  "source_id": "b5453b47-07b0-5ad7-beb3-42b79396bb76",
  "target_id

### 2. Phân tích chi tiết 1 file + kiểm chứng Stable ID (idempotency)

Gọi module `src/task2/verify_stable_id.py` để kiểm tra việc reprocess cùng 1 nội dung file có tạo ra node/edge ID trùng khớp tuyệt đối không.

In [11]:
def run_task2_verify_stable_id():
    ok = test_stable_id()
    if not ok:
        print('[CANH BAO] Stable ID verification khong thanh cong.')

run_task2_verify_stable_id()

Sample file: docs/source/_config.py
Nodes  run1=5 run2=5 | ID overlap = 5/5
Edges  run1=6 run2=6 | ID overlap = 6/6
Metadata id run1=fbc8c700-7469-5400-898f-bd8616625351
Metadata id run2=fbc8c700-7469-5400-898f-bd8616625351
Metadata ID identical? True

[OK] Stable ID verified — reprocess cung noi dung khong tao ID moi.
[OK] ID uniqueness: nodes=5 edges=6 dangling=0

Edge type counts: {'AST': 4, 'CFG': 2}

--- SAMPLE NODE ---
{
  "id": "b5453b47-07b0-5ad7-beb3-42b79396bb76",
  "type": "Module",
  "label": "Module",
  "file_path": "docs/source/_config.py",
  "ast_path": "Module@0:0-0:0",
  "start_line": null,
  "start_column": null,
  "end_line": null,
  "end_column": null,
  "code": "",
  "properties": {},
  "schema_version": "1.0.0",
  "event_time": "2026-07-24T13:00:06.586928Z"
}

--- SAMPLE EDGE ---
{
  "id": "5a01cb7a-4332-5df7-8964-570b8f7935a6",
  "source_id": "b5453b47-07b0-5ad7-beb3-42b79396bb76",
  "target_id": "b8030daa-556d-5866-a3e1-dba25a7acfaa",
  "type": "AST",
  "schema_

### 3. Reflection – Task 2

**What worked**
- Dùng `ast.NodeVisitor` trích xuất đủ 4 loại quan hệ (AST, CFG, DFG, CALL) độc lập, không phụ thuộc tool ngoài C/C++.
- UUIDv5 tính từ `(file_path, location, node_class)` đảm bảo **deterministic**: reprocess 100 lần vẫn ra đúng ID đó.
- Thiết kế bounded memory (parse từng file) hoạt động mượt với repo lớn (`peft`).

**What failed**
- Ban đầu DFG chỉ track được variable assignment đơn giản, chưa xử lý hết tuple unpacking (`a, b = x, y`).

**How resolved**
- Bổ sung helper duyệt target nodes của `ast.Assign` và `ast.AnnAssign` trong `cpg_parser.py`.

## **Task 3: Kafka Topic Design**

### 0. Schema thiết kế (message contract)

#### Topic `cpg-nodes` 
| Field | Kiểu | Mô tả |
|:---|:---|:---|
| `id` | string (UUIDv5) | Stable node id — **Kafka key** |
| `type` / `label` | string | Loại AST node (`FunctionDef`, `Name`, ...) |
| `file_path` | string | Đường dẫn tương đối (POSIX) |
| `start_line`, `start_column`, `end_line`, `end_column` | int/null | Vị trí nguồn |
| `code` | string | Snippet |
| `properties` | object | name/value phụ |
| `schema_version` | string | vd. `1.0.0` |
| `event_time` | string | ISO-8601 UTC |

#### Topic `cpg-edges` 
| Field | Kiểu | Mô tả |
|:---|:---|:---|
| `id` | string (UUIDv5) | Stable edge id — **Kafka key** |
| `source_id`, `target_id` | string | Liên kết tới node ids |
| `type` | string | `AST` / `CFG` / `DFG` / `CALL` |
| `properties` | object (optional) | vd. `callee` cho CALL |
| `schema_version`, `event_time` | string | Bắt buộc |

#### Topic `cpg-metadata` 
| Field | Kiểu | Mô tả |
|:---|:---|:---|
| `id` | string (UUIDv5) | Stable file id — **Kafka key** |
| `file_path` | string | Đường dẫn tương đối |
| `size_bytes`, `num_lines` | int | Kích thước file |
| `sha256` | string | Hash nội dung |
| `processed_at`, `status` | string | Thời điểm / SUCCESS |
| `schema_version`, `event_time` | string | Bắt buộc |

#### Topic `cpg-errors` 
| Field | Kiểu | Mô tả |
|:---|:---|:---|
| `id` | string (UUIDv5) | `uuid5(file_path + ":error")` |
| `file_path`, `error_type`, `error_message`, `stack_trace` | string | Chi tiết lỗi |
| `occurred_at`, `schema_version`, `event_time` | string | Bắt buộc |

**Cấu hình broker (lab):** `replication-factor=1`, `partitions=1` (single-broker WSL).


### 0. Cấu hình

Import các module trong `src/task3/` — broker mặc định `localhost:9092` (Docker Compose Task 4 hoặc Kafka WSL).

In [12]:
from src.task3.setup_kafka import setup_kafka
from src.task3.emit import emit
from src.task3.verify import verify
from src.task3.config import KAFKA_BOOTSTRAP, TOPICS, DEMO_LIMIT

print(f"Kafka bootstrap : {KAFKA_BOOTSTRAP}")
print(f"Topics          : {', '.join(TOPICS)}")
print(f"Emit limit      : {DEMO_LIMIT} files")

Kafka bootstrap : 127.0.0.1:9092
Topics          : cpg-nodes, cpg-edges, cpg-metadata, cpg-errors
Emit limit      : 5 files


### 1. Khởi động Kafka & tạo 4 topic

Chạy `src/task3/setup_kafka.py` — tự khởi động ZooKeeper + Kafka (Docker) nếu cần, rồi tạo 4 topic bắt buộc.

In [13]:
def run_task3_setup_kafka():
    ok = setup_kafka()
    if not ok:
        print('[CANH BAO] Chua tao duoc topic — hay khoi dong Kafka (Docker/WSL) roi chay lai cell nay.')

run_task3_setup_kafka()

[INFO] Kafka chua san sang — khoi dong ZooKeeper + Kafka (Docker Compose)...
time="2026-07-24T20:00:15+07:00" level=warning msg="D:\\_STUDY\\BigData\\Lab\\Lab4\\code\\spark-streaming-lab\\src\\task4\\docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
 Container cpg-zookeeper  Created
 Container cpg-kafka  Created
 Container cpg-zookeeper  Starting
 Container cpg-zookeeper  Started
 Container cpg-kafka  Starting
 Container cpg-kafka  Started
[INFO] Cho Kafka Broker san sang (15 giay)...
[INFO] Tao 4 topic bat buoc tren broker Kafka...
  cpg-nodes: da ton tai
  cpg-edges: Created topic cpg-edges.
  cpg-metadata: Created topic cpg-metadata.
  cpg-errors: Created topic cpg-errors.

[INFO] kafka-topics --list
  - cpg-edges
  - cpg-errors
  - cpg-metadata
  - cpg-nodes
[OK] Du 4 topic Task 3 tren broker.


### 2. Emit sự kiện từ Parser Service lên Kafka (bằng chứng end-to-end)

Gọi `src/task3/emit.py` để phát dữ liệu lên Kafka topics.

In [14]:
def run_task3_emit():
    ok = emit()
    if not ok:
        print('[CANH BAO] Chua emit duoc — dam bao setup_kafka da thanh cong.')

run_task3_emit()

CMD: d:\Anaconda3\envs\min_ds-env\python.exe D:\_STUDY\BigData\Lab\Lab4\code\spark-streaming-lab\src\task2\cpg_parser.py --limit 5 --repo-root D:\_STUDY\BigData\Lab\Lab4\code\spark-streaming-lab\peft --discovered-csv D:\_STUDY\BigData\Lab\Lab4\code\spark-streaming-lab\output\discovered_files.csv --bootstrap-servers 127.0.0.1:9092 --schema-version 1.0.0
------------------------------------------------------------------------
=== Starting CPG Parser Service ===
Discovered CSV : D:\_STUDY\BigData\Lab\Lab4\code\spark-streaming-lab\output\discovered_files.csv
Repo Root      : D:\_STUDY\BigData\Lab\Lab4\code\spark-streaming-lab\peft
Kafka Brokers  : 127.0.0.1:9092
Schema Version : 1.0.0
Dry Run Mode   : False

Found 5 source files to parse.
[OK] Connected to Kafka Producer successfully.

[1/5] Processing: docs\source\_config.py ... Parsed and Sent (Nodes: 5, Edges: 6)
[2/5] Processing: examples\adamss_finetuning\glue_adamss_asa_example.py ... Parsed and Sent (Nodes: 2222, Edges: 3935)
[3/5] 

### 3. Kiểm chứng topic offset & mẫu message trên Kafka

Gọi `src/task3/verify.py` để kiểm tra message count và sample JSON payload từ Kafka.

In [15]:
def run_task3_verify():
    ok = verify()
    if not ok:
        print('[CANH BAO] Chua verify duoc — dam bao emit da thanh cong.')

run_task3_verify()

=== Kafka topic offsets (bang chung ghi thanh cong) ===
  cpg-nodes       begin=0        end=7948     messages=7948
  cpg-edges       begin=0        end=13950    messages=13950
  cpg-metadata    begin=0        end=5        messages=5
  cpg-errors      begin=0        end=0        messages=0
[OK] Saved -> D:\_STUDY\BigData\Lab\Lab4\code\spark-streaming-lab\output\task3_offsets.csv

===== SAMPLE from cpg-nodes =====
Kafka key: b5453b47-07b0-5ad7-beb3-42b79396bb76
{
  "id": "b5453b47-07b0-5ad7-beb3-42b79396bb76",
  "type": "Module",
  "label": "Module",
  "file_path": "docs/source/_config.py",
  "ast_path": "Module@0:0-0:0",
  "start_line": null,
  "start_column": null,
  "end_line": null,
  "end_column": null,
  "code": "",
  "properties": {},
  "schema_version": "1.0.0",
  "event_time": "2026-07-24T13:00:51.955263Z"
}

===== SAMPLE from cpg-edges =====
Kafka key: 5a01cb7a-4332-5df7-8964-570b8f7935a6
{
  "id": "5a01cb7a-4332-5df7-8964-570b8f7935a6",
  "source_id": "b5453b47-07b0-5ad7-beb3

### 4. Reflection – Task 3

**What worked**
- 4 topic tách rõ theo loại event → Neo4j chỉ cần subscribe nodes/edges, Spark/Mongo chỉ cần metadata.
- `schema_version` + `event_time` trên mọi message; Kafka key = stable id hỗ trợ sink idempotent ở Task 4.
- `advertised.listeners=PLAINTEXT://localhost:9092` giúp producer Windows nói chuyện được với broker trong WSL.

**What failed**
- Lần đầu broker advertise hostname WSL (`DESKTOP-....localdomain`) khiến client Windows kết nối metadata fail.

**How resolved**
- Sửa `advertised.listeners=PLAINTEXT://127.0.0.1:9092` trong `src/task4/docker-compose.yml` và gom logic tao topic vao `src/task3/setup_kafka.py`.

## **Task 4: Graph Topology Ingestion into Neo4j**

### Mục tiêu
- Kết nối **Neo4j Kafka Connector Sink** với hai topic `cpg-nodes` và `cpg-edges`.
- Ghi trực tiếp cấu trúc đồ thị từ **Kafka** vào **Neo4j**, không thông qua Spark.
- Sử dụng `MERGE` thay cho `CREATE` để đảm bảo **idempotent**, tức là khi xử lý lại dữ liệu sẽ không tạo bản ghi trùng.

### Kiến trúc

```text
cpg_parser.py  -->  Kafka (cpg-nodes, cpg-edges)  -->  Kafka Connect  -->  Neo4j
                    localhost:9092                      :8083               :7687
```

### Docker Compose

| Dịch vụ | Image | Cổng |
|:---|:---|:---:|
| ZooKeeper | confluentinc/cp-zookeeper:7.6.1 | 2181 |
| Kafka | confluentinc/cp-kafka:7.6.1 | 9092 |
| Kafka Connect | confluentinc/cp-kafka-connect:7.6.1 | 8083 |
| Neo4j | neo4j:5.20.0-community | 7474, 7687 |

### 0. Cấu hình đường dẫn

In [16]:

import subprocess as _sp
from pathlib import Path

# Chuyển đường dẫn Windows sang định dạng WSL
def to_wsl_path(p):
    p = str(p).replace("\\", "/")
    if len(p) >= 2 and p[1] == ":":
        return f"/mnt/{p[0].lower()}{p[2:]}"
    return p

# Tìm Python của Conda trong WSL
def find_conda_wsl():
    for name in ["miniconda3", "anaconda3", "miniforge3"]:
        path = f"~/{name}/bin/python3"
        r = _sp.run(
            ["wsl", "-e", "bash", "-c", f"ls {path}"],
            capture_output=True,
            text=True,
        )
        if r.returncode == 0:
            return r.stdout.strip()

    # Nếu không tìm thấy thì dùng Python mặc định
    return "python3"


WSL_ROOT = to_wsl_path(PROJECT_ROOT)
TASK4_WSL = f"{WSL_ROOT}/src/task4"
CONDA_WSL = find_conda_wsl()

print(f"WSL_ROOT  = {WSL_ROOT}")
print(f"TASK4_WSL = {TASK4_WSL}")
print(f"CONDA_WSL = {CONDA_WSL}")

def run_wsl(bash_cmd, timeout=180):
    """Chạy lệnh bash trong WSL, trả về CompletedProcess."""
    return subprocess.run(
        ["wsl", "-e", "bash", "-c", bash_cmd],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        timeout=timeout,
    )

WSL_ROOT  = /mnt/d/_STUDY/BigData/Lab/Lab4/code/spark-streaming-lab
TASK4_WSL = /mnt/d/_STUDY/BigData/Lab/Lab4/code/spark-streaming-lab/src/task4
CONDA_WSL = python3


### 1. Khởi động các dịch vụ của Task 4

In [17]:
def start_task4_services():
    r = run_wsl(f'{CONDA_WSL} {TASK4_WSL}/start.py', timeout=400)
    if r.stdout: print(r.stdout)
    if r.returncode != 0: print('STDERR:', r.stderr[:500])

start_task4_services()

Kiểm tra và xóa container cũ (nếu có)...
  Đã xóa container: cpg-zookeeper (trạng thái trước đó: running)
  Đã xóa container: cpg-kafka (trạng thái trước đó: running)
  Đã xóa container: cpg-kafka-connect (trạng thái trước đó: exited)
  Đã xóa container: cpg-neo4j (trạng thái trước đó: exited)
Khởi động Docker Stack...
  Container cpg-zookeeper  Started
  Container cpg-neo4j  Started
  Container cpg-kafka  Started
  Container cpg-kafka-connect  Started
Chờ Kafka Broker sẵn sàng (15 giây)...
Tạo các Kafka topic...
  cpg-nodes: Created topic cpg-nodes.
  cpg-edges: Created topic cpg-edges.
  cpg-metadata: Created topic cpg-metadata.
  cpg-errors: Created topic cpg-errors.
Chờ Kafka Connect và plugin Neo4j khởi động (có thể mất 3–5 phút)...
  [1/36] Chờ 5 giây...
  [2/36] Chờ 5 giây...
  [3/36] Chờ 5 giây...
  [4/36] Chờ 5 giây...
  [5/36] Chờ 5 giây...
  [6/36] Chờ 5 giây...
  [7/36] Chờ 5 giây...
  Đã tìm thấy plugin Neo4j: org.neo4j.connectors.kafka.sink.Neo4jConnector
Docker Stack và 

### 2. Đăng ký Neo4j Connector Sink

Sử dụng Kafka Connect REST API để đăng ký hai connector:

- `neo4j-cpg-nodes-sink`: đọc dữ liệu từ topic `cpg-nodes` và ghi các node vào Neo4j bằng câu lệnh `MERGE`.
- `neo4j-cpg-edges-sink`: đọc dữ liệu từ topic `cpg-edges` và ghi các relationship vào Neo4j bằng câu lệnh `MERGE`.

In [18]:
def register_neo4j_connectors():
    r = run_wsl(f'{CONDA_WSL} {TASK4_WSL}/setup.py')
    if r.stdout: print(r.stdout)
    if r.returncode != 0: print('STDERR:', r.stderr[:500])

register_neo4j_connectors()

Đang chờ Kafka Connect sẵn sàng... OK

Kiểm tra plugin Neo4j:
  Tìm thấy: org.neo4j.connectors.kafka.sink.Neo4jConnector (phiên bản 5.1.10)
  Tìm thấy: org.neo4j.connectors.kafka.source.Neo4jConnector (phiên bản 5.1.10)

Đăng ký các connector...
  [neo4j-cpg-nodes-sink] Đăng ký thành công
  [neo4j-cpg-edges-sink] Đăng ký thành công

Chờ connector khởi động (30 giây)...

Trạng thái connector:
  neo4j-cpg-nodes-sink: RUNNING, tasks=['RUNNING']
  neo4j-cpg-edges-sink: RUNNING, tasks=['RUNNING']



### 3. Gửi CPG Events lên Kafka

Chạy `cpg_parser.py` với 5 tệp Python đầu tiên để phát (emit) các sự kiện lên các Kafka topic sau:

- `cpg-nodes`: chứa các node của Code Property Graph (AST, CFG, DFG).
- `cpg-edges`: chứa các cạnh giữa các node, bao gồm quan hệ AST (cha–con), CFG (luồng điều khiển), DFG (luồng dữ liệu) và `CALL`.
- `cpg-metadata`: chứa thông tin metadata của từng tệp Python được xử lý.

In [19]:
def emit_cpg_events():
    r = run_wsl(f'{CONDA_WSL} {TASK4_WSL}/emit.py', timeout=300)
    if r.stdout: print(r.stdout)
    if r.returncode != 0: print('STDERR:', r.stderr[:500])

emit_cpg_events()

=== Starting CPG Parser Service ===
Discovered CSV : /mnt/d/_STUDY/BigData/Lab/Lab4/code/spark-streaming-lab/output/discovered_files.csv
Repo Root      : /mnt/d/_STUDY/BigData/Lab/Lab4/code/spark-streaming-lab/peft
Kafka Brokers  : localhost:9092
Schema Version : 1.0.0
Dry Run Mode   : False

Found 5 source files to parse.
[OK] Connected to Kafka Producer successfully.

[1/5] Processing: docs/source/_config.py ... Parsed and Sent (Nodes: 5, Edges: 6)
[2/5] Processing: examples/adamss_finetuning/glue_adamss_asa_example.py ... Parsed and Sent (Nodes: 2222, Edges: 3935)
[3/5] Processing: examples/adamss_finetuning/glue_adamss_asa_manual_example.py ... Parsed and Sent (Nodes: 2175, Edges: 3841)
[4/5] Processing: examples/adamss_finetuning/image_classification_adamss_asa.py ... Parsed and Sent (Nodes: 2319, Edges: 4123)
[5/5] Processing: examples/alora_finetuning/alora_finetuning.py ... Parsed and Sent (Nodes: 1227, Edges: 2045)

=== Parser Run Summary ===
Total Source Files processed : 5
S

### 4. Kiểm chứng dữ liệu đã được đưa vào Neo4j

Sau khi các sự kiện được gửi lên Kafka, Kafka Connect sẽ tự động đọc dữ liệu từ các topic và ghi vào Neo4j.

Cell dưới đây sẽ chờ khoảng **90 giây** để Kafka Connect xử lý dữ liệu theo từng lô (batch), sau đó truy vấn Neo4j và hiển thị kết quả.

**Các nội dung cần kiểm tra:**

1. Số lượng `CpgNode` lớn hơn 0.
2. Số lượng `CPG_EDGE` lớn hơn 0, bao gồm các loại quan hệ: `AST`, `CFG`, `DFG` và `CALL`.
3. `duplicates = 0`, xác nhận câu lệnh `MERGE` hoạt động đúng và không tạo dữ liệu trùng lặp khi xử lý lại.

In [20]:
def verify_neo4j_ingestion():
    r = run_wsl(f'{CONDA_WSL} {TASK4_WSL}/verify.py')
    if r.stdout: print(r.stdout)
    if r.returncode != 0: print('STDERR:', r.stderr[:500])

verify_neo4j_ingestion()

Tổng số CpgNode  : 8,274
Tổng số CPG_EDGE : 14,447

Thống kê loại node (Top 10):
  Load                         1,837
  Name                         1,529
  Constant                     1,387
  Attribute                    475
  Call                         460
  keyword                      414
  Store                        329
  alias                        214
  Expr                         212
  Assign                       204

Thống kê loại relationship:
  AST                          8,268
  CFG                          5,215
  DFG                          589
  CALL                         375

Các tệp đã xử lý:
  examples/adamss_finetuning/image_classification_adamss_asa.py  ->  2,319 node
  examples/adamss_finetuning/glue_adamss_asa_example.py  ->  2,222 node
  examples/adamss_finetuning/glue_adamss_asa_manual_example.py  ->  2,175 node
  examples/alora_finetuning/alora_finetuning.py  ->  1,227 node
  src/peft/__init__.py  ->  326 node
  docs/source/_config.py  ->  5 node

N

### 5. Kết quả

**1. Triển khai hạ tầng**  
Kiểm tra trạng thái các container trong stack:

<div align="center">
  <img src="result/task4/1_v2.png" alt="Docker status" width="800"/>
  <p><i>Hình 4.5.1: Các dịch vụ Docker (Zookeeper, Kafka, Connect, Neo4j) đang hoạt động (Up)</i></p>
</div>  

**2. Trạng thái Connectors**  
Xác nhận các sink connector đã được đăng ký và đang chạy:

<div align="center">
  <img src="result/task4/2_v2.png" alt="Connector status" width="800"/>
  <p><i>Hình 4.5.2: Trạng thái RUNNING của neo4j-cpg-nodes-sink và neo4j-cpg-edges-sink</i></p>
</div>  

**3. Kiểm chứng dữ liệu trong Neo4j**  
Dưới đây là kết quả thống kê số lượng Nodes và Relationships đã được ingestion thành công:

<div align="center">
  <img src="result/task4/3.2_v2.png" alt="Neo4j Nodes" width="800"/>
  <p><i>Hình 4.5.3: Kết quả truy vấn đếm CpgNode</i></p>
</div>

<div align="center">
  <img src="result/task4/3.1_v2.png" alt="Neo4j Edges" width="800"/>
  <p><i>Hình 4.5.4: Kết quả truy vấn đếm CPG_EDGE</i></p>
</div>

### 6. Reflection

**What worked**
- Docker Compose giúp khởi động toàn bộ stack (ZooKeeper, Kafka, Kafka Connect và Neo4j) chỉ với một lệnh, giúp việc thiết lập môi trường nhanh và thuận tiện.
- Neo4j Kafka Connector Sink tự động đọc dữ liệu từ các topic `cpg-nodes` và `cpg-edges`, sau đó thực thi câu lệnh Cypher để ghi trực tiếp vào Neo4j mà không cần thông qua Spark.
- Sử dụng `MERGE` thay cho `CREATE` đảm bảo tính idempotent. Sau khi emit dữ liệu nhiều lần, số lượng node và relationship không thay đổi, đồng thời `duplicates = 0`.

**What failed**
- Neo4j Connector phiên bản 5.x thay đổi khóa cấu hình từ `neo4j.topic.cypher.<name>` sang `neo4j.cypher.topic.<name>`, khiến connector không hoạt động khi sử dụng cấu hình của phiên bản cũ.
- Connector ghi `cpg-edges` chỉ hoạt động khi các node tương ứng đã tồn tại trong Neo4j. Nếu node connector chưa xử lý xong, các relationship sẽ không được tạo.
- Đường dẫn tệp sinh trên Windows sử dụng dấu `\`, trong khi các script chạy trên WSL yêu cầu định dạng `/`, dẫn đến lỗi không tìm thấy tệp.

**How resolved**
- Cập nhật cấu hình connector theo đúng cú pháp của Neo4j Connector 5.x và kiểm tra plugin thông qua Kafka Connect REST API trước khi đăng ký connector.
- Khởi động Kafka Connect trước, chờ connector sẵn sàng rồi mới emit dữ liệu để đảm bảo các node được ghi vào Neo4j trước khi xử lý relationship.
- Chuyển đổi đường dẫn từ định dạng Windows sang định dạng WSL trước khi truyền vào các script, giúp các bước xử lý hoạt động ổn định trên cả hai môi trường.

## **Task 5: Spark Structured Streaming -> MongoDB**

### Mục tiêu
- Sử dụng **Spark Structured Streaming** đọc topic `cpg-metadata` từ Kafka.
- Parse JSON payload theo schema đã định nghĩa.
- Ghi kết quả vào **MongoDB** collection `peft_db.source_metadata` ở chế độ **append**.
- Dùng **checkpoint** để đảm bảo fault-tolerance và exactly-once delivery.

### Thiết kế

| Thành phần | Giá trị |
|:---|:---|
| **Kafka topic** | `cpg-metadata` |
| **MongoDB URI** | `mongodb://127.0.0.1:27017` |
| **Database** | `peft_db` |
| **Collection** | `source_metadata` |
| **Checkpoint** | `checkpoints/task5_metadata/` |
| **Trigger** | `processingTime=10 seconds` |

### Kiến trúc

```text
cpg_parser.py  -->  Kafka (cpg-metadata)  -->  Spark Structured Streaming  -->  MongoDB
                    localhost:9092              pyspark 3.5.0 / 10s trigger      peft_db.source_metadata
                                                       |
                                                       v
                                          checkpoints/task5_metadata/
```


### 1. Khởi động MongoDB (Docker)

In [23]:
import subprocess, sys, time
from pathlib import Path

TASK5_DIR = Path.cwd() / 'src' / 'task5'

print('[INFO] Khởi động MongoDB container...')
r = subprocess.run(
    ['docker', 'compose', 'up', '-d'],
    cwd=str(TASK5_DIR), capture_output=True, text=True
)
print(r.stdout or r.stderr)

# Kiểm tra container
r2 = subprocess.run(
    ['docker', 'ps', '--filter', 'name=mongo', '--format', 'table {{.Names}}\t{{.Status}}'],
    capture_output=True, text=True
)
print(r2.stdout)

[INFO] Khởi động MongoDB container...
time="2026-07-24T20:06:35+07:00" level=warning msg="d:\\_STUDY\\BigData\\Lab\\Lab4\\code\\spark-streaming-lab\\src\\task5\\docker-compose.yml: the attribute `version` is obsolete, it will be ignored, please remove it to avoid potential confusion"
 Container cpg-mongodb  Running

NAMES         STATUS
cpg-mongodb   Up 2 minutes



### 2. Chạy Spark Structured Streaming (WSL)

> ⚠️ **Lưu ý CỰC KỲ QUAN TRỌNG:**
> Cell dưới đây chạy **VÔ TẬN** (sẽ bị treo bằng dấu `[*]`) vì Spark Streaming liên tục lắng nghe Kafka.
> **KHÔNG ĐƯỢC TẮT CELL NÀY.**
> Để chạy Task 6, bạn cần **mở thêm một tab/cửa sổ Jupyter Notebook mới** (nhân bản tab hiện tại).
> Hoặc bạn có thể chạy terminal riêng lệnh: `!wsl python3 -m src.task5.ingest` 


### Hướng dẫn chạy Spark Streaming (WSL)

> ⚠️ **Lưu ý:** Lệnh `!wsl` khi chạy trong Jupyter Notebook có thể bị treo hoặc không hiển thị log theo thời gian thực. Vì vậy, bạn bắt buộc phải chạy nó ở một **Terminal riêng**.

**Các bước thực hiện:**
1. Mở cửa sổ Terminal Ubuntu (WSL) trên Windows.
2. Di chuyển vào thư mục bài lab:
   ```bash
   cd /mnt/d/_STUDY/BigData/Lab/Lab4/code/spark-streaming-lab
   ```
3. Khởi chạy tiến trình Streaming:
   ```bash
   python3 -m src.task5.ingest
   ```

> Tiến trình này sẽ chạy liên tục để lắng nghe Kafka. **Đừng tắt cửa sổ Terminal này**, hãy thu nhỏ nó xuống và chạy tiếp Task 6 ở bên dưới.

### 3. Kiểm chứng dữ liệu trong MongoDB

In [24]:
from pymongo import MongoClient
import json, time

MONGO_URI  = 'mongodb://127.0.0.1:27017'
DB_NAME    = 'peft_db'
COLL_NAME  = 'source_metadata'

print(f'[INFO] Kết nối MongoDB: {MONGO_URI}/{DB_NAME}.{COLL_NAME}')
client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
coll   = client[DB_NAME][COLL_NAME]

count = coll.count_documents({})
print(f'[OK] Số documents trong collection: {count}')

if count > 0:
    sample = coll.find_one({}, {'_id': 0})
    print('\nDocument mẫu:')
    print(json.dumps(sample, indent=2, default=str))
else:
    print('[WARN] Chưa có dữ liệu. Đảm bảo spark-submit đang chạy và emit.py đã được chạy trước.')

client.close()

[INFO] Kết nối MongoDB: mongodb://127.0.0.1:27017/peft_db.source_metadata
[OK] Số documents trong collection: 6

Document mẫu:
{
  "file_path": "docs/source/_config.py",
  "event_time": "2026-07-24 13:03:21.951000",
  "id": "fbc8c700-7469-5400-898f-bd8616625351",
  "num_lines": 7,
  "processed_at": "2026-07-24 13:03:21.951000",
  "schema_version": "1.0.0",
  "sha256": "62bc147b19a03f310e079c3f03ada33aaf2d4f8c43a62a89d3917c2d54ebe0e2",
  "size_bytes": 287,
  "status": "SUCCESS"
}


### 4. Reflection

**What worked**
- Spark Structured Streaming với trigger `processingTime=10s` hoạt động ổn định, đọc topic `cpg-metadata` và ghi vào MongoDB thông qua Docker trên Windows.
- Thay vì ghi đúp dữ liệu (append thuần túy), cơ chế **Upsert** (`operationType=update` theo `file_path`) đảm bảo mỗi file chỉ tồn tại duy nhất một document mới nhất.
- Giải quyết triệt để các rắc rối liên quan tới `HADOOP_HOME` và `winutils.exe` bằng cách chuyển toàn bộ tiến trình chạy Spark sang môi trường máy ảo Linux (WSL).

**What failed**
- Khi chạy bằng Python trên Windows (Anaconda), Spark báo lỗi thiếu `HADOOP_HOME` và file `winutils.exe`.
- Khi chuyển sang WSL, Spark báo lỗi `TimeoutException` không kết nối được với Kafka Broker. Nguyên nhân do cấu hình `localhost:9092` làm WSL nhầm lẫn và ưu tiên sử dụng mạng IPv6 (`::1`) thay vì IPv4.
- Việc dùng lệnh `!wsl` để chạy ngầm tiến trình vô tận (long-running) của Spark bên trong cell của Jupyter Notebook làm treo cell và không hiển thị log theo thời gian thực.

**How resolved**
- Chạy Spark Streaming hoàn toàn native trên môi trường WSL để bypass các rắc rối về Hadoop File System của Windows.
- Sửa cấu hình Kafka Bootstrap Server thành IP cứng `127.0.0.1:9092` để ép hệ thống dùng mạng IPv4, giúp WSL giao tiếp thành công với container Kafka nằm trên Docker Windows.
- Đưa tiến trình Spark Streaming ra chạy ở một màn hình Terminal (Ubuntu) riêng biệt, giải phóng Jupyter Notebook để tiếp tục chạy Task 6.

## **Task 6: Idempotent Replay Verification**

### Mục tiêu
- Chứng minh pipeline xử lý **idempotent**: emit lại cùng sự kiện không tạo duplicate trong Neo4j/MongoDB.
- Kiểm chứng **3 invariant**:
  - **[A]** Neo4j: không có `CpgNode` trùng sau khi replay (MERGE hoạt động đúng).
  - **[B]** MongoDB: document được **update** (processed_at mới hơn) không phải insert mới.
  - **[C]** Spark Checkpoint: offset được ghi nhận → đảm bảo exactly-once.

### Quy trình

```
mutate.py  →  replay.py  →  (Spark xử lý)  →  verify.py
```
| Bước | Script | Mô tả |
|:--:|:--|:--|
| 1 | `mutate.py` | Sửa 1 file trong peft (thêm comment + timestamp) để thay đổi SHA-256 |
| 2 | `replay.py` | Gọi lại `cpg_parser.py` cho file đó → emit lại events lên Kafka |
| 3 | `verify.py` | Kiểm chứng Neo4j/MongoDB/Checkpoint không có duplicate |

### Kiến trúc

```text
mutate.py  -->  peft/src/peft/__init__.py  (SHA-256 thay đổi)
                             |
                             v
replay.py  -->  cpg_parser.py  -->  Kafka (cpg-nodes, cpg-edges, cpg-metadata)
                                        |                       |
                                        v                       v
                              Neo4j MERGE                Spark --> MongoDB update
                              (no duplicate)             (processed_at mới hơn)
                                        |                       |
                                        +----> verify.py <------+
                                               [PASS] A, B, C
```


### E2E Auto Verification (Strict Mode)

Chạy kịch bản tự động hóa kiểm tra tính idempotent. Kịch bản này sẽ:
1. Kiểm tra trạng thái Database trước (Báo FAIL nếu rỗng).
2. Tự động thêm comment (Mutate) vào file `src/peft/__init__.py`.
3. Đẩy lại file vào Kafka (Replay Lần 1).
4. Đẩy lại file vào Kafka (Replay Lần 2) để test duplicate.
5. Kiểm tra kết quả (Strict Assertions).

> ⚠️ **Yêu cầu:** Bạn phải đang để Cell của Task 5 chạy ngầm ở một tab khác!


In [25]:
def verify_neo4j_ingestion():
    print("[INFO] Đang chạy E2E Automation Verification bên trong máy ảo WSL...")
    # Gọi module verify.py bằng WSL
    r = run_wsl(f"{CONDA_WSL} -m src.task6.verify", timeout=600)
    
    if r.stdout: print(r.stdout)
    if r.returncode != 0: print("STDERR:", r.stderr[:1000])

verify_neo4j_ingestion()


[INFO] Đang chạy E2E Automation Verification bên trong máy ảo WSL...
Task 6 — Strict Idempotent Replay Verification (E2E)
[*] File được chọn để test: src/peft/__init__.py

[*] Đang lấy trạng thái BEFORE...
    - Neo4j: 326 nodes, 497 edges
    - Mongo: 1 docs, SHA=50dc4386..., Time=2026-07-24 12:45:38.674000
    - Checkpoint: batch 0

[*] Bước 1: Mutating file (Thêm chú thích để đổi SHA)...
[OK] Đã mutate file: /mnt/d/_STUDY/BigData/Lab/Lab4/code/spark-streaming-lab/peft/src/peft/__init__.py
     Timestamp patch : 2026-07-24T13:07:33Z

[*] Bước 2: Replay Lần 1 (Đẩy sự kiện lên Kafka)...
    -> Đợi 15s cho Spark Streaming kịp Ingest vào MongoDB...

[*] Bước 3: Replay Lần 2 (Đẩy lại sự kiện giống hệt Lần 1 để test Trùng lặp)...
    -> Đợi 15s cho Spark Streaming kịp xử lý...

[*] Đang lấy trạng thái AFTER...
    - Neo4j: 330 nodes, 504 edges
    - Mongo: 1 docs, SHA=2c8a1b4b..., Time=2026-07-24 13:07:54.312000
    - Checkpoint: batch 2

[*] So sánh và kết luận (Strict Mode)...
  [PASS] N

### 4. Reflection

**What worked**
- `MERGE` trong Neo4j Cypher đảm bảo không tạo node trùng khi replay — chứng minh tính idempotent của pipeline đầu cuối.
- Stable UUID (UUIDv5 từ file path + AST path) giúp cùng một node luôn có cùng ID dù được parse bao nhiêu lần.
- Spark checkpoint ghi nhận offset sau mỗi micro-batch, đảm bảo không xử lý lại message cũ sau khi restart.

**What failed**
- Kafka `NoBrokersAvailable` trên Windows Docker Desktop do auto-detect API version timeout. Fix bằng `api_version=(2, 6, 0)`.
- Checkpoint path dạng relative (`./checkpoints/`) gây lỗi `NativeIO$Windows.access0` trên Windows. Fix bằng absolute path.

**How resolved**
- Thêm `api_version=(2, 6, 0)` vào `KafkaProducer` trong `cpg_parser.py` để bypass handshake timeout.
- Dùng `Path(__file__).resolve()` để tạo absolute checkpoint path, tránh lỗi Windows native IO.